# Exercice 18 - Streaming Spark Avance

## Objectifs
- Comprendre les watermarks et le traitement tardif
- Maitriser les differents modes de trigger
- Implementer des jointures en streaming
- Gerer les checkpoints pour la tolerance aux pannes

---

## 1. Concepts avances du streaming

```
+------------------------------------------------------------------+
|                  STREAMING SPARK AVANCE                          |
+------------------------------------------------------------------+
|                                                                  |
|  WATERMARK : Gestion des donnees tardives                        |
|  +---------------------------------------------------------+    |
|  |                                                         |    |
|  |  Temps reel    Watermark         Donnees acceptees      |    |
|  |      |            |                    |                |    |
|  |   12:05        12:00               >= 12:00             |    |
|  |      |<-- 5min -->|                                     |    |
|  |                                                         |    |
|  |  Si delai = 5 min, les donnees avec timestamp < 12:00   |    |
|  |  seront ignorees (arrivees trop tard)                   |    |
|  +---------------------------------------------------------+    |
|                                                                  |
|  TRIGGERS : Frequence de traitement                              |
|  +---------------------------------------------------------+    |
|  |  - processingTime("10 seconds") : toutes les 10 sec     |    |
|  |  - once()                       : une seule fois        |    |
|  |  - continuous("1 second")       : latence minimale      |    |
|  |  - availableNow()               : traite tout dispo     |    |
|  +---------------------------------------------------------+    |
|                                                                  |
|  CHECKPOINTS : Tolerance aux pannes                              |
|  +---------------------------------------------------------+    |
|  |  Sauvegarde :                                           |    |
|  |  - Offsets Kafka traites                                |    |
|  |  - Etat des aggregations                                |    |
|  |  - Metadata du stream                                   |    |
|  +---------------------------------------------------------+    |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, window, expr,
    count, sum as spark_sum, avg, max as spark_max, min as spark_min,
    current_timestamp, lit, struct, to_json
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, ArrayType, TimestampType
)
import time

spark = SparkSession.builder \
    .appName("StreamingAvance") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

KAFKA_BROKER = "broker:9092"
CHECKPOINT_PATH = "s3a://bronze/checkpoints"

print(f"Spark version: {spark.version}")

## 3. Watermarks - Gestion des donnees tardives

In [ ]:
# Schema des commandes
commande_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("total", DoubleType(), True),
    StructField("status", StringType(), True)
])

# Lire le stream Kafka
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load()

print("Stream configure")

In [ ]:
# Parser et ajouter watermark
df_parsed = df_stream \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        col("timestamp").alias("kafka_time")
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total",
        to_timestamp("data.timestamp").alias("event_time"),
        "kafka_time"
    ) \
    .withWatermark("event_time", "5 minutes")  # Tolere 5 min de retard

print("Watermark defini : 5 minutes")

In [ ]:
# Aggregation avec fenetre glissante et watermark
df_windowed = df_parsed \
    .groupBy(
        window(col("event_time"), "2 minutes", "1 minute"),  # Fenetre 2min, glisse 1min
        "customer_id"
    ) \
    .agg(
        count("*").alias("nb_commandes"),
        spark_sum("total").alias("total_ventes"),
        spark_max("total").alias("max_commande")
    )

print("Aggregation fenetre glissante configuree")

In [ ]:
# Lancer le stream avec mode update
query_watermark = df_windowed \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream avec watermark demarre")

In [ ]:
# Arreter apres 30 secondes
time.sleep(30)
query_watermark.stop()
print("Stream arrete")

## 4. Differents modes de trigger

In [ ]:
# Mode once() - Traitement unique
# Utile pour le traitement batch incremental

df_once = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select("data.*")

query_once = df_once \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(once=True) \
    .start()

query_once.awaitTermination()
print("Traitement once() termine")

In [ ]:
# Mode availableNow() - Traite tout ce qui est disponible
# Similaire a once() mais avec meilleure gestion des partitions

df_available = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select("data.order_id", "data.customer_id", "data.total")

query_available = df_available \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(availableNow=True) \
    .start()

query_available.awaitTermination()
print("Traitement availableNow() termine")

## 5. Checkpoints - Tolerance aux pannes

In [ ]:
# Stream avec checkpoint
df_checkpoint = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        col("timestamp").alias("kafka_time")
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total",
        "kafka_time"
    )

print("Stream avec checkpoint prepare")

In [ ]:
# Ecrire avec checkpoint dans MinIO
query_ckpt = df_checkpoint \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "s3a://bronze/streaming/commandes") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/commandes") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream avec checkpoint demarre")
print(f"Checkpoint: {CHECKPOINT_PATH}/commandes")

In [ ]:
# Verifier le statut
print("Statut du stream:")
print(f"  ID: {query_ckpt.id}")
print(f"  Run ID: {query_ckpt.runId}")
print(f"  Actif: {query_ckpt.isActive}")
print(f"  Status: {query_ckpt.status}")

In [ ]:
# Attendre et arreter
time.sleep(20)
query_ckpt.stop()
print("Stream arrete - checkpoint sauvegarde")

## 6. Jointure Stream-Static

In [ ]:
# Creer un DataFrame statique de reference
# Simule une table de clients

clients_data = [
    ("CUST-001", "Alice Martin", "Paris", "VIP"),
    ("CUST-002", "Bob Dupont", "Lyon", "Standard"),
    ("CUST-003", "Claire Leroy", "Marseille", "Premium"),
    ("CUST-004", "David Moreau", "Toulouse", "Standard"),
    ("CUST-005", "Emma Petit", "Nice", "VIP"),
]

df_clients = spark.createDataFrame(
    clients_data,
    ["customer_id", "nom", "ville", "segment"]
)

df_clients.show()

In [ ]:
# Stream de commandes
df_commandes_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total"
    )

print("Stream de commandes pret")

In [ ]:
# Jointure stream-static
df_enrichi = df_commandes_stream.join(
    df_clients,
    on="customer_id",
    how="left"
).select(
    "order_id",
    "customer_id",
    "nom",
    "ville",
    "segment",
    "total"
)

print("Jointure stream-static configuree")

In [ ]:
# Executer la jointure
query_join = df_enrichi \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream avec jointure demarre")

In [ ]:
time.sleep(20)
query_join.stop()
print("Stream arrete")

## 7. Ecriture vers Kafka (Stream to Stream)

In [ ]:
# Lire, transformer et ecrire vers un autre topic
df_transform = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select(
        col("data.customer_id").alias("key"),
        struct(
            col("data.order_id"),
            col("data.customer_id"),
            col("data.total"),
            (col("data.total") * 0.2).alias("tva"),
            current_timestamp().alias("processed_at")
        ).alias("value")
    ) \
    .select(
        col("key"),
        to_json(col("value")).alias("value")
    )

print("Transformation preparee")

In [ ]:
# Ecrire vers un nouveau topic Kafka
query_kafka = df_transform \
    .writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("topic", "commandes-enrichies") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/kafka-to-kafka") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream Kafka -> Kafka demarre")

In [ ]:
time.sleep(20)
query_kafka.stop()
print("Stream arrete")

## 8. Gestion de plusieurs streams

In [ ]:
# Lister tous les streams actifs
streams = spark.streams.active

print(f"Nombre de streams actifs: {len(streams)}")
for stream in streams:
    print(f"  - {stream.name}: {stream.id}")

In [ ]:
# Arreter tous les streams
for stream in spark.streams.active:
    stream.stop()
    print(f"Stream {stream.id} arrete")

print("Tous les streams arretes")

---

## Exercice

**Objectif** : Creer un pipeline streaming complet

**Consigne** :
1. Lisez le topic "logs-application" en streaming
2. Ajoutez un watermark de 2 minutes
3. Calculez le nombre de logs par niveau (INFO, WARNING, ERROR) par fenetre de 1 minute
4. Ecrivez les resultats dans la console

A vous de jouer :

In [ ]:
# TODO: Definir le schema des logs

In [ ]:
# TODO: Lire le stream avec watermark

In [ ]:
# TODO: Aggreger par niveau et fenetre

---

## Resume

Dans ce notebook, vous avez appris :
- Comment utiliser les **watermarks** pour gerer les donnees tardives
- Les differents **modes de trigger** (processingTime, once, availableNow)
- Comment configurer les **checkpoints** pour la tolerance aux pannes
- Comment faire des **jointures stream-static**
- Comment **ecrire vers Kafka** depuis un stream

### Prochaine etape
Dans le prochain notebook, nous construirons un pipeline complet Bronze/Silver/Gold.